# MLOps in Practice: Training Pipelines, Experiment Tracking & Model Evaluation

**Author:** Laela Zorana | [github.com/LaelaZorana](https://github.com/LaelaZorana) | [github.com/LaelaZorana/mlops-training-pipeline](https://github.com/LaelaZorana/mlops-training-pipeline)

---

## The Core MLOps Problems

Every ML practitioner eventually hits the same wall:

- **Reproducibility:** "The model I trained last Tuesday had better accuracy. What was different?" → *You didn't log the hyperparameters.*
- **Experiment comparison:** "Is learning rate 0.001 better than 0.0001?" → *You can't tell because the runs aren't tracked.*
- **Production readiness:** "How fast does inference run at p99?" → *You never benchmarked it.*

This notebook demonstrates a **self-contained MLOps workflow** that solves these problems without requiring MLflow, W&B, or any external service:

1. A clean PyTorch training loop with structured epoch logging
2. A lightweight JSONL experiment tracker (zero dependencies)
3. Multi-experiment comparison with visualizations
4. Rigorous model evaluation (accuracy, precision, recall, F1, latency)
5. Distributed training configuration reference
6. PyTorch vs JAX training step comparison

All code patterns are from [`mlops-training-pipeline`](https://github.com/LaelaZorana/mlops-training-pipeline).

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

import json
import time
import os
import tempfile
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')
torch.manual_seed(42)
np.random.seed(42)

---
## Section 1: Building a Clean PyTorch Training Loop

A production training loop should:
- Return structured data per epoch (not just print to stdout)
- Track both training and validation metrics
- Log throughput (samples/sec) to detect resource bottlenecks
- Be reproducible via explicit random seeding

We train a binary classifier on synthetic data.

In [ ]:
@dataclass
class EpochResult:
    epoch: int
    train_loss: float
    val_loss: float
    train_acc: float
    val_acc: float
    samples_per_sec: float
    epoch_time_s: float


class BinaryClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def make_dataset(n=2000, n_features=20, noise=0.1):
    X = torch.randn(n, n_features)
    w = torch.randn(n_features)
    logits = X @ w + noise * torch.randn(n)
    y = (logits > 0).float()
    split = int(0.8 * n)
    return (TensorDataset(X[:split], y[:split]),
            TensorDataset(X[split:], y[split:]))


def compute_accuracy(logits: torch.Tensor, labels: torch.Tensor) -> float:
    preds = (torch.sigmoid(logits) > 0.5).float()
    return (preds == labels).float().mean().item()


def train_model(
    lr: float = 1e-3,
    n_epochs: int = 15,
    batch_size: int = 64,
    seed: int = 42,
    verbose: bool = True,
) -> List[EpochResult]:
    torch.manual_seed(seed)
    train_ds, val_ds = make_dataset()
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)

    model = BinaryClassifier(input_dim=20).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    history = []
    if verbose:
        print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10} {'Train Acc':>10} {'Val Acc':>9} {'Samp/s':>9} {'Time':>7}")
        print("-" * 72)

    for epoch in range(1, n_epochs + 1):
        t0 = time.perf_counter()
        model.train()
        train_losses, train_accs, n_samples = [], [], 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
            train_accs.append(compute_accuracy(out.detach(), yb))
            n_samples += len(xb)

        model.eval()
        val_losses, val_accs = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                out = model(xb)
                val_losses.append(criterion(out, yb).item())
                val_accs.append(compute_accuracy(out, yb))

        epoch_time = time.perf_counter() - t0
        result = EpochResult(
            epoch=epoch,
            train_loss=float(np.mean(train_losses)),
            val_loss=float(np.mean(val_losses)),
            train_acc=float(np.mean(train_accs)),
            val_acc=float(np.mean(val_accs)),
            samples_per_sec=n_samples / epoch_time,
            epoch_time_s=epoch_time,
        )
        history.append(result)

        if verbose:
            print(f"{epoch:>6} {result.train_loss:>12.4f} {result.val_loss:>10.4f} "
                  f"{result.train_acc:>10.4f} {result.val_acc:>9.4f} "
                  f"{result.samples_per_sec:>9.0f} {result.epoch_time_s:>6.3f}s")

    return history, model


print("Training with lr=1e-3 (Experiment 1)...")
history_1, model_1 = train_model(lr=1e-3, n_epochs=15, verbose=True)

---
## Section 2: Experiment Tracking with JSONL

MLflow and W&B are great, but sometimes you need a zero-dependency solution — during a hackathon, in a restricted environment, or when prototyping. JSONL (newline-delimited JSON) is:
- Human-readable
- Appendable (no file locking issues)
- Queryable with standard Python
- Git-diffable

Each epoch gets one JSON line. Each experiment gets one file. Load them all to compare.

In [ ]:
EXPERIMENT_DIR = Path(tempfile.mkdtemp(prefix="mlops_experiments_"))
print(f"Experiment directory: {EXPERIMENT_DIR}")


class ExperimentTracker:
    def __init__(self, exp_dir: Path):
        self.exp_dir = exp_dir
        self.exp_dir.mkdir(parents=True, exist_ok=True)

    def log_experiment(
        self,
        run_name: str,
        hparams: dict,
        history: List[EpochResult],
    ):
        path = self.exp_dir / f"{run_name}.jsonl"
        with open(path, 'w') as f:
            # First line: hyperparameters
            f.write(json.dumps({"type": "hparams", "run": run_name, **hparams}) + "\n")
            # One line per epoch
            for ep in history:
                f.write(json.dumps({"type": "epoch", "run": run_name, **asdict(ep)}) + "\n")
        print(f"Logged {len(history)} epochs → {path.name}")

    def load_run(self, run_name: str) -> Dict:
        path = self.exp_dir / f"{run_name}.jsonl"
        hparams, epochs = {}, []
        with open(path) as f:
            for line in f:
                rec = json.loads(line)
                if rec["type"] == "hparams":
                    hparams = rec
                elif rec["type"] == "epoch":
                    epochs.append(rec)
        return {"hparams": hparams, "epochs": epochs}


tracker = ExperimentTracker(EXPERIMENT_DIR)

# Log experiment 1 (already trained)
tracker.log_experiment("run_lr_1e-3", {"lr": 1e-3, "batch_size": 64}, history_1)

# Train and log experiment 2
print("\nTraining with lr=1e-2 (Experiment 2)...")
history_2, model_2 = train_model(lr=1e-2, n_epochs=15, verbose=False)
tracker.log_experiment("run_lr_1e-2", {"lr": 1e-2, "batch_size": 64}, history_2)

# Train and log experiment 3
print("Training with lr=1e-4 (Experiment 3)...")
history_3, model_3 = train_model(lr=1e-4, n_epochs=15, verbose=False)
tracker.log_experiment("run_lr_1e-4", {"lr": 1e-4, "batch_size": 64}, history_3)

print("\nAll 3 experiments logged.")

---
## Section 3: Comparing Experiments

Load all runs from disk, plot training curves, and print a comparison table.

In [ ]:
run_names = ["run_lr_1e-3", "run_lr_1e-2", "run_lr_1e-4"]
runs = {name: tracker.load_run(name) for name in run_names}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = ['#2196F3', '#F44336', '#4CAF50']
labels = ['lr=1e-3 (baseline)', 'lr=1e-2 (high LR)', 'lr=1e-4 (low LR)']

for ax, metric, title in zip(axes, ['val_loss', 'val_acc'], ['Validation Loss', 'Validation Accuracy']):
    for (name, run), color, label in zip(runs.items(), colors, labels):
        epochs = [ep['epoch'] for ep in run['epochs']]
        values = [ep[metric] for ep in run['epochs']]
        ax.plot(epochs, values, color=color, label=label, linewidth=2.5, marker='o', markersize=4)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(metric.replace('_', ' ').title())
    ax.set_title(f'{title} — All 3 Experiments')
    ax.legend()

plt.tight_layout()
plt.show()

# Comparison table
def find_convergence_epoch(epochs, metric='val_loss', threshold=0.05):
    """Epoch at which val_loss first drops below threshold above final value."""
    final = min(ep[metric] for ep in epochs)
    for ep in epochs:
        if ep[metric] <= final + threshold:
            return ep['epoch']
    return len(epochs)

print("\n" + "=" * 80)
print(f"  EXPERIMENT COMPARISON SUMMARY")
print("=" * 80)
print(f"{'Run':<18} {'LR':>8} {'Best Val Loss':>14} {'Best Val Acc':>13} {'Conv. Epoch':>12}")
print("-" * 80)
for name, run, label in zip(run_names, runs.values(), labels):
    epochs = run['epochs']
    best_loss = min(ep['val_loss'] for ep in epochs)
    best_acc = max(ep['val_acc'] for ep in epochs)
    conv = find_convergence_epoch(epochs)
    lr = run['hparams']['lr']
    print(f"{name:<18} {lr:>8.0e} {best_loss:>14.4f} {best_acc:>13.4f} {conv:>12}")
print("=" * 80)

---
## Section 4: Model Evaluation

Accuracy alone is insufficient. A model with 95% accuracy on a 95/5 imbalanced dataset is just predicting the majority class. Always report precision, recall, F1, and inference latency.

In [ ]:
_, val_ds = make_dataset()
val_loader = DataLoader(val_ds, batch_size=64)

best_model = model_1  # lr=1e-3 (typically best)
best_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        logits = best_model(xb.to(DEVICE))
        preds = (torch.sigmoid(logits) > 0.5).float().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

y_pred = np.array(all_preds)
y_true = np.array(all_labels)

acc = (y_pred == y_true).mean()
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

# Inference latency benchmark
single_input = torch.randn(1, 20).to(DEVICE)
latencies_ms = []
with torch.no_grad():
    for _ in range(500):
        t0 = time.perf_counter()
        best_model(single_input)
        latencies_ms.append((time.perf_counter() - t0) * 1000)

lat = np.array(latencies_ms)

print("=" * 55)
print("  MODEL EVALUATION REPORT (best model: lr=1e-3)")
print("=" * 55)
print(f"  Accuracy : {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1       : {f1:.4f}")
print("-" * 55)
print(f"  Inference Latency (single sample, n=500)")
print(f"    p50  : {np.percentile(lat, 50):.3f} ms")
print(f"    p95  : {np.percentile(lat, 95):.3f} ms")
print(f"    p99  : {np.percentile(lat, 99):.3f} ms")
print(f"    mean : {lat.mean():.3f} ms")
print("=" * 55)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Class 0', 'Class 1'])
disp.plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix')

# Latency histogram
axes[1].hist(lat, bins=50, color='#2196F3', edgecolor='white', alpha=0.85)
for pct, color, label in [(50, 'green', 'p50'), (95, 'orange', 'p95'), (99, 'red', 'p99')]:
    v = np.percentile(lat, pct)
    axes[1].axvline(v, color=color, linestyle='--', linewidth=2, label=f'{label}={v:.2f}ms')
axes[1].set_xlabel('Latency (ms)')
axes[1].set_ylabel('Count')
axes[1].set_title('Single-Sample Inference Latency Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Section 5: Distributed Training Configuration

Choosing the right distributed training strategy depends on model size, hardware, and memory budget. This section provides a practical reference.

| Strategy | When to Use | Key Tradeoff |
|----------|------------|-------------|
| **DDP** (DistributedDataParallel) | Models that fit on 1 GPU | Fast, simple, all-reduce gradients |
| **FSDP** (Fully Sharded DP) | Models > single GPU memory | Memory efficient, more communication overhead |
| **Model Parallelism** | Extremely large models (100B+) | Complex to implement correctly |

In [ ]:
def estimate_model_memory_gb(
    n_params: int,
    batch_size: int,
    seq_len: int = 512,
    precision: str = "fp16",
) -> float:
    """Rough memory estimate: params + gradients + optimizer states + activations."""
    bytes_per_param = {"fp32": 4, "fp16": 2, "bf16": 2, "int8": 1}[precision]
    param_bytes = n_params * bytes_per_param
    # Gradient: same size as params
    grad_bytes = n_params * bytes_per_param
    # Adam optimizer: 2 moments (fp32 each) per param
    optimizer_bytes = n_params * 4 * 2
    # Activations: rough estimate (batch_size * seq_len * hidden_approx)
    hidden_approx = max(512, int(np.sqrt(n_params / 12)))
    activation_bytes = batch_size * seq_len * hidden_approx * bytes_per_param * 2
    total = (param_bytes + grad_bytes + optimizer_bytes + activation_bytes) / 1e9
    return total


model_configs = [
    ("1B",  1_000_000_000),
    ("7B",  7_000_000_000),
    ("13B", 13_000_000_000),
    ("70B", 70_000_000_000),
]
batch_sizes = [1, 4, 8]
precisions = ["fp32", "bf16", "fp16"]

print("MEMORY ESTIMATION TABLE (GB) — Training Memory per GPU")
print("Strategy recommendation: DDP if <40GB/GPU, FSDP otherwise")
print()
print(f"{'Model':<8} {'Precision':<10} " + " ".join(f"{'BS='+str(b):>10}" for b in batch_sizes))
print("-" * (8 + 10 + 3 + 12 * len(batch_sizes)))

for model_name, n_params in model_configs:
    for precision in precisions:
        row = f"{model_name:<8} {precision:<10} "
        for bs in batch_sizes:
            mem = estimate_model_memory_gb(n_params, bs, precision=precision)
            flag = " ⚠" if mem > 40 else "  "
            row += f"{mem:>8.1f}GB{flag}"
        print(row)
    print()

print("⚠ = Exceeds 40GB (A100 80GB can handle; smaller GPUs need FSDP or model parallelism)")
print()
print("DDP Config snippet:")
print("  model = nn.parallel.DistributedDataParallel(model, device_ids=[local_rank])")
print()
print("FSDP Config snippet (for 7B+ models):")
print("  from torch.distributed.fsdp import FullyShardedDataParallel as FSDP")
print("  model = FSDP(model, auto_wrap_policy=transformer_auto_wrap_policy,")
print("               mixed_precision=MixedPrecision(param_dtype=torch.bfloat16))")

---
## Section 6: JAX vs PyTorch — Side-by-Side Training Step

PyTorch and JAX represent fundamentally different programming models:

- **PyTorch** is **imperative**: you write a training step as normal Python. Side effects (parameter updates) happen in place. Easy to debug.
- **JAX** is **functional**: functions must be pure (no side effects). State is passed explicitly. `jax.jit` compiles the function. More complex, but enables XLA optimizations and elegant parallelism with `jax.pmap`.

Below: the same training step in both frameworks.

In [ ]:
pytorch_training_step = '''
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  PYTORCH: Imperative training step
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def train_step(model, optimizer, batch):
    x, y = batch
    
    optimizer.zero_grad()          # Clear accumulated gradients
    logits = model(x)              # Forward pass
    loss = F.cross_entropy(        # Compute loss
        logits, y
    )
    loss.backward()                # Backprop (in-place grad accumulation)
    
    torch.nn.utils.clip_grad_norm_(
        model.parameters(), 1.0    # Gradient clipping
    )
    optimizer.step()               # Update parameters IN PLACE
    
    return loss.item()             # Return scalar

# Usage:
for batch in dataloader:
    loss = train_step(model, optimizer, batch)
    # model.parameters() are silently mutated
'''

jax_training_step = '''
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  JAX: Functional training step (pure function)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@jax.jit  # Compile with XLA — first call traces, subsequent calls fast
def train_step(state, batch):
    x, y = batch
    
    def loss_fn(params):           # Inner fn: params → loss
        logits = state.apply_fn(
            {'params': params}, x  # Forward pass, no side effects
        )
        return optax.softmax_cross_entropy_with_integer_labels(
            logits, y
        ).mean()
    
    loss, grads = jax.value_and_grad(  # Compute loss AND gradients
        loss_fn
    )(state.params)                    # in one pass (efficient)
    
    new_state = state.apply_gradients( # Returns NEW state object
        grads=grads                    # Original state unchanged
    )
    return new_state, loss

# Usage:
for batch in dataloader:
    state, loss = train_step(state, batch)
    # state is a NEW object; old state still accessible if needed
'''

print(pytorch_training_step)
print(jax_training_step)

print("\n" + "=" * 70)
print("  PYTORCH vs JAX TRADEOFFS")
print("=" * 70)
tradeoffs = [
    ("Programming Model", "Imperative (easier debug)", "Functional (pure fns)"),
    ("State Management",  "In-place mutation",          "Explicit state passing"),
    ("JIT Compilation",   "torch.compile (optional)",   "jax.jit (standard)"),
    ("Parallelism",       "DDP / FSDP",                 "jax.pmap (elegant)"),
    ("Debugging",         "Easy (Python debugger)",     "Hard (traced graphs)"),
    ("Ecosystem",         "Massive (HuggingFace etc.)", "Growing (Flax, Optax)"),
    ("Recommended for",   "Most production use cases",  "Research, TPU workloads"),
]
print(f"{'Dimension':<20} {'PyTorch':^25} {'JAX':^25}")
print("-" * 70)
for dim, pt, jx in tradeoffs:
    print(f"{dim:<20} {pt:<25} {jx:<25}")
print("=" * 70)

---
## Conclusion

This notebook demonstrated a complete, self-contained MLOps workflow:

| Component | Implementation | Key Benefit |
|-----------|---------------|-------------|
| Training loop | EpochResult dataclass + structured logging | Structured, queryable history |
| Experiment tracking | JSONL files per run | Zero dependencies, git-diffable |
| Experiment comparison | Load + plot all runs | Visual identification of best config |
| Model evaluation | Accuracy + Precision + Recall + F1 + latency | Production-readiness assessment |
| Distributed training | DDP vs FSDP memory table | Right strategy for model size |
| PyTorch vs JAX | Side-by-side code | Clear tradeoffs for framework choice |

---
*By Laela Zorana | [github.com/LaelaZorana](https://github.com/LaelaZorana) | [github.com/LaelaZorana/mlops-training-pipeline](https://github.com/LaelaZorana/mlops-training-pipeline)*